# Project 6: elektriciteit per persoon in Suriname en drie buurlanden

**De vraag.** Hoe verhoudt het elektriciteitsverbruik per persoon in Suriname zich tot dat van Guyana, Trinidad en Tobago en Jamaica? Verandert de volgorde als je het totaal van elk land deelt door zijn bevolking?

**De data.** De energiedataset van Our World in Data, die de elektriciteitsvraag overneemt uit de Yearly Electricity Data van Ember (2026) en de bevolking uit een eigen bevolkingsreeks. Our World in Data heeft het bestand voor het laatst bijgewerkt op 2026-09-30, en de kopie voor de cursus is gedownload op 2026-11-06. De tabel houdt de vier landen vanaf 2000.

**Je team.** Vier rollen wisselen elke week: bouwer, datacontroleur, AI-controleur en presentator. Deze week vergelijkt de datacontroleur elk cijfer per persoon met de eigen kolom van de uitgever.

**Wat je vrijdag inlevert.** Dit notebook met elke CHECK geslaagd, één grafiek met twee panelen, een notitie van ongeveer 200 woorden voor een ambtenaar van het ministerie die de elektriciteitsvoorziening van Suriname plant, en jullie AI-logboek.

## Laad de data

Deze cel krijg je cadeau. Hij leest de tabel uit de cursusmap, of rechtstreeks bij Our World in Data als het bestand er niet is.

In [ ]:
import pandas as pd

URL = "https://catalog.ourworldindata.org/energy/owid_energy/owid_energy.csv"
COLUMNS = ['country', 'iso_code', 'year', 'population', 'electricity_demand_twh', 'electricity_demand_per_capita_kwh']
COUNTRIES = ['SUR', 'GUY', 'TTO', 'JAM']
try:
    energy = pd.read_csv("../data/week06-electricity.csv")
except FileNotFoundError:
    raw = pd.read_csv(URL, usecols=COLUMNS)
    energy = (raw[raw["iso_code"].isin(COUNTRIES) & (raw["year"] >= 2000)][COLUMNS]
              .sort_values(["country", "year"]).reset_index(drop=True))
energy

## TODO 1: beschrijf de tabel

Druk het aantal rijen, de landen, het eerste en laatste jaar en de ontbrekende waarden per kolom af. Sla daarna in `latest` het laatste jaar op waarin alle vier landen een waarde voor de elektriciteitsvraag hebben. Kijk naar het laatste jaar in de tabel voordat je beslist.

In [ ]:
# TODO 1
latest = None

In [ ]:
# CHECK 1
counts = energy.dropna(subset=["electricity_demand_twh"]).groupby("year")["country"].nunique()
expected_latest = int(counts[counts == 4].index.max())
assert latest is not None, "latest is nog leeg"
assert int(latest) == expected_latest, "latest moet het laatste jaar zijn waarin alle vier landen een waarde voor de vraag hebben"
print("CHECK 1 geslaagd:", expected_latest)

## TODO 2: rangschik op totaal verbruik

Sla in `rank_total` een lijst op van de vier landen, van de hoogste elektriciteitsvraag in `latest` naar de laagste.

In [ ]:
# TODO 2
rank_total = None

In [ ]:
# CHECK 2
in_latest = energy[energy["year"] == expected_latest].set_index("country")
total_order = list(in_latest["electricity_demand_twh"].sort_values(ascending=False).index)
assert rank_total is not None, "rank_total is nog leeg"
assert list(rank_total) == total_order
print("CHECK 2 geslaagd:", total_order)

## TODO 3: kilowattuur per persoon

Voeg een kolom `kwh_per_person` toe die de elektriciteitsvraag van elke rij deelt door de bevolking in dezelfde rij. De vraag staat in terawattuur (TWh), en één TWh is 1.000.000.000 kilowattuur (kWh).

De tabel bevat ook het eigen cijfer van de uitgever, `electricity_demand_per_capita_kwh`. De CHECK vergelijkt jouw cijfer daarmee.

In [ ]:
# TODO 3

In [ ]:
# CHECK 3
assert "kwh_per_person" in energy.columns, "voeg eerst de kolom kwh_per_person toe"
own = energy["electricity_demand_twh"] * 1e9 / energy["population"]
assert ((energy["kwh_per_person"] - own).abs() < 0.01).sum() == own.notna().sum(), "deel elke rij door de bevolking in dezelfde rij"
gap = (energy["kwh_per_person"] - energy["electricity_demand_per_capita_kwh"]).abs().max()
assert gap < 1, "je cijfers wijken af van die van de uitgever: controleer de omrekening van de eenheden"
print("CHECK 3 geslaagd: het grootste verschil met het cijfer van de uitgever is", round(gap, 3), "kWh")

## TODO 4: rangschik per persoon, en de jaren met een andere volgorde

Sla in `rank_per_person` de vier landen op, van het hoogste aantal kWh per persoon in `latest` naar het laagste. Sla in `moved` de landen op waarvan de plaats verschilt van hun plaats in `rank_total`.

Sla daarna in `order_years` elk jaar op waarin de volgorde per persoon van de vier landen verschilt van de volgorde in `latest`.

In [ ]:
# TODO 4
rank_per_person = None
moved = None
order_years = None

In [ ]:
# CHECK 4
pp = energy.assign(pp=energy["electricity_demand_twh"] * 1e9 / energy["population"]).dropna(subset=["pp"])
def order_in(year):
    return list(pp[pp["year"] == year].sort_values("pp", ascending=False)["country"])
assert rank_per_person is not None and moved is not None and order_years is not None, "een van de drie is nog leeg"
assert list(rank_per_person) == order_in(expected_latest)
assert sorted(moved) == sorted(c for c in total_order if total_order.index(c) != order_in(expected_latest).index(c))
expected_years = [int(y) for y in sorted(pp["year"].unique()) if order_in(y) != order_in(expected_latest)]
assert sorted(int(y) for y in order_years) == expected_years
print("CHECK 4 geslaagd:", order_in(expected_latest), sorted(moved), expected_years)

## TODO 5: één grafiek, twee panelen

Maak één figuur met twee staafdiagrammen naast elkaar voor `latest`: de totale vraag in TWh links en kWh per persoon rechts. Houd de landen in beide panelen in dezelfde volgorde, geef elk paneel een titel en een benoemde as, en vermeld de bron op de figuur.

In [ ]:
# TODO 5

## TODO 6: groei sinds 2000

Sla in `growth_total` het percentage op waarmee de totale vraag van Suriname groeide van 2000 tot `latest`, en in `growth_per_person` het percentage voor het aantal kWh per persoon. Druk de bevolkingsgroei van Suriname over dezelfde jaren af, en leg in één zin uit waarom de twee groeicijfers verschillen.

In [ ]:
# TODO 6
growth_total = None
growth_per_person = None

In [ ]:
# CHECK 6
sur = energy[energy["country"] == "Suriname"].set_index("year")
sur_kwh = sur["electricity_demand_twh"] * 1e9 / sur["population"]
assert growth_total is not None and growth_per_person is not None, "growth_total of growth_per_person is nog leeg"
assert abs(growth_total - (sur.loc[expected_latest, "electricity_demand_twh"] / sur.loc[2000, "electricity_demand_twh"] - 1) * 100) < 0.1
assert abs(growth_per_person - (sur_kwh[expected_latest] / sur_kwh[2000] - 1) * 100) < 0.1
print("CHECK 6 geslaagd:", round(growth_total, 1), "procent in totaal en", round(growth_per_person, 1), "procent per persoon")

## TODO 7: de AI-controle

Schrijf een functie `check_claim(country, year, claimed_kwh)` die teruggeeft hoe ver een beweerd cijfer in kWh per persoon afligt van het cijfer in de tabel voor dat land en dat jaar, als percentage van het cijfer in de tabel. Een bewering die 10% te hoog is, geeft 10 terug.

Vraag daarna je AI-assistent, zonder hem de tabel te laten zien: *"Rangschik Suriname, Guyana, Trinidad en Tobago en Jamaica naar elektriciteitsverbruik per persoon. Geef voor elk land het cijfer in kWh, het jaar waarvoor het geldt en de bevolking waardoor je deelde."* Schrijf het antwoord in jullie AI-logboek.

Toets elk cijfer met `check_claim` voor het jaar dat de AI noemde. Vergelijk elke bevolking die de AI gebruikte met de bevolking in de tabel voor dat jaar, en vergelijk de volgorde met `rank_per_person` en `order_years`. Noteer of de AI de landen op totaal verbruik rangschikte terwijl je naar verbruik per persoon vroeg.

In [ ]:
# TODO 7
check_claim = None

In [ ]:
# CHECK 7
assert check_claim is not None, "check_claim is nog leeg"
sur = energy[energy["country"] == "Suriname"].set_index("year")
right = sur.loc[expected_latest, "electricity_demand_twh"] * 1e9 / sur.loc[expected_latest, "population"]
assert abs(check_claim("Suriname", expected_latest, right)) < 0.01
wrong = sur.loc[expected_latest, "electricity_demand_twh"] * 1e9 / sur.loc[2000, "population"]
assert abs(check_claim("Suriname", expected_latest, wrong) - (wrong / right - 1) * 100) < 0.01
print("CHECK 7 geslaagd: delen door de bevolking van 2000 maakt het cijfer van Suriname", round((wrong / right - 1) * 100, 1), "procent te hoog")

## TODO 8: de notitie voor een ambtenaar

Schrijf in een tekstcel hieronder ongeveer 200 woorden voor een ambtenaar van het ministerie die de elektriciteitsvoorziening van Suriname plant. Vergelijk Suriname met de drie andere landen in totaal en per persoon, gebruik de getallen die je hebt berekend, noem de bron en zeg één ding dat de data de ambtenaar niet kan vertellen.